# Machine learning (AIC-4101C)

The $k$-means' algorithm (also called Lloyd's algorithm) computes a partition of a set **S** of observations into $k$ subsets. It is composed of an initialization step (randomly choosing an initial partition, or equivalently, the barycenters of each subsets), and two repeated steps until convergence of the method (the partition does not change) :

- **Assignment** : each observation is associated to the partition of the closest barycenter 

$${\large S_i^{(t)} = \left\{\mathbf{x}_j : \|\mathbf{x}_j - \mathbf{m}_i^{(t)}Â \| \leq \|\mathbf{x}_j - \mathbf{m}_{i^*}^{(t)} \|, \forall i^* = 1, \dots, k \right\} }$$

- **Update** : recompute the barycenter of each partition $${\large\mathbf{m}_i^{(t+1)} = \frac{1}{|Â S_i^{(t)} |} \sum_{\mathbf{x_j} \in S_i^{(t)}} \mathbf{x}_j}$$

This algorithm finds a local minimum of the cost function $${\large \sum_{i=1}^k \sum_{\mathbf{x}_j \in S_i} \| \mathbf{x}_j - m_i \|^2}$$
This function represents the global error when each observation is replaced by the barycenter of its class.

>Show that both steps minimize the cost function

- Assigment : by definition it minimizes all the distances.
- Update : we have seen that the mean value, that is $\mathbf{m}$, minimizes the euclidean distance. For example, we did that for Linear regression.

>Show that the algorithm converges. 

The number of possible permutations is finite.

>Implement the functions of each step of the algorithm :
- initialisation (take the $k$ first points as barycenters)
- assignment
- update
- loop

The loop function should save the cost function of each iteration, so that you can plot its evolution.

In [1]:
from mllab import *


Packages:
    numpy as np
    matplotlib.pyplot as plt

Functions:
    plotXY
    add_bias
    grad_descent
    MSE
    fmin_bfgs
    plot_frontiere
    time



In [11]:
def dist(x,y):
    return ((x-y)**2).sum()

def init (k,x):
    return x[:k]

def assign(x,m):
    n=len(x)
    k=len(m)
    y=[]
    S = dict([(i,[]) for i in range(k)])
    
    for xx in x:
        D = np.zeros(k)
        for i in range(k):
            D[i] = dist (xx, m[i])
            ki = np.argmin(D)
            S[ki].append(xx)
            y.append(ki)
        return S,y
    
    
def update(S):
    k = len(S)
    m = [np.mean(S_i, axis = 0) for _,S_i in S.items()]
    for S_i in S.items():
        
        print(S_i)
    return m

def loop (k,x,m0):
    converged = False
    while not converged:
        S,y = assign (x, m0)
        m = update (S)
        converged = np.prod(m==m0)
        m0 = m
    _,y = assign (x,m)
    return np.array(m),np.array(y)


def cost(m,S):
    c=0
    for i,mi in enumerate(m):
        for x in S[i]:
            c+= dist (x,mi)
    return c
            
        

>Apply your method on generated data and check that your cost function is decreasing.

In [12]:
from sklearn.datasets import make_blobs
k=3
X,_ = make_blobs(n_features=2, centers = k)
M0 = init(k,X)

M,y,C = loop(k,X,M0)
plt.plot(C)
plt.show()
plot(x,y)

(0, [array([8.80092733, 6.14069589]), array([8.80092733, 6.14069589]), array([8.80092733, 6.14069589])])
(1, [])
(2, [])
(0, [array([8.80092733, 6.14069589])])
(1, [array([8.80092733, 6.14069589]), array([8.80092733, 6.14069589])])
(2, [])


ValueError: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()

The quality of the local minimum depends on the initial choice of barycenters : we know that the minimum can be arbitrarily bad and that the worst case is reached in practice. Different strategies have been proposed.

The simplest one is the one we implemented : choose the $k$ first points of our set.

>Show with a simple example that we can converge to a bad local minimum.

The most used method is to choose randomly those $k$ points amongst the data set.

>Write an initialization function using that strategy. Is this method better that the previous one ? How would you improve it ?

The $k$-means++ strategy gives us a warranty of approximate our local minimum in a $O(\log k)$ time :
- Choose a first barycenter $c_1$ randomly.
- Given the $p$-first barycenters $c_1, \dots, c_p$, choose point $x$ to be barycenter $c_{p+1}$ with probability ${\large \frac{D(x)}{\sum_{i=1}^n D(x_i)}}$, where $D(x) = \min_{1 \leq i \leq p}Â \| x - c_i \|$

>Write an initialization function using that method.

>Compare the cost functions with the two different methods.